In [21]:
# ─── Make sure you’ve installed imbalanced-learn: ───────────────────────────────
# pip install imbalanced-learn

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# 1. Load your data
df = pd.read_csv('ai4i2020.csv')  # adjust path if needed

# 2. Original class distribution
print("Original class counts:")
print(df['Machine failure'].value_counts(), "\n")

# 3. Define features & target
feature_cols = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]
X = df[feature_cols]
y = df["Machine failure"]

# 4. Under-sample the full dataset to exactly 339 of each class
rus_full = RandomUnderSampler(sampling_strategy={0:339, 1:339}, random_state=42)
X_bal, y_bal = rus_full.fit_resample(X, y)

# 5. Verify balanced counts
print("Balanced class counts:")
print(y_bal.value_counts(), "\n")

# 6. Split into train/test (your test set remains balanced but untouched by sampler)
X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

# 7. Preprocessor: scale only the numeric features
preprocessor = ColumnTransformer([
    ("scale", StandardScaler(), feature_cols)
])

# 8. Build pipeline: scaler + classifier
pipeline = Pipeline([
    ("scale", preprocessor),
    ("clf", RandomForestClassifier(n_estimators=100, random_state=42))
])

# 9. Fit on the BALANCED training set
pipeline.fit(X_train, y_train)

# 10. Evaluate on the untouched TEST set
y_pred = pipeline.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))


Original class counts:
Machine failure
0    9661
1     339
Name: count, dtype: int64 

Balanced class counts:
Machine failure
0    339
1    339
Name: count, dtype: int64 

Test accuracy: 0.9117647058823529

Classification report:
               precision    recall  f1-score   support

           0       0.91      0.91      0.91        68
           1       0.91      0.91      0.91        68

    accuracy                           0.91       136
   macro avg       0.91      0.91      0.91       136
weighted avg       0.91      0.91      0.91       136

